In [ ]:
# SKKU PubMed Research Lineage — ONE CELL
# 1) SKKU affiliation PubMed 전체 수집 (10k 제한 회피용 연/월 분할 + pagination)
# 2) 전체 seed corpus에서 연구자 profile / coauthor network / research community 생성
# 3) ORCID 기반으로 일부 핵심 연구자의 SKKU 이후 후속 논문 심층 추적
# 4) 결과 CSV/JSON/Markdown/HTML 생성 + 주요 표 표시

import os, sys, subprocess, json, shutil
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
from google.colab import drive

# ===== 사용자 설정 =====
START_YEAR = 2010
END_YEAR = 2026
FULL_CRAWL = True        # True: 연/월 분할 pagination으로 기간 전체 수집
MAX_RESULTS = 0          # FULL_CRAWL=True일 때 0=전체, 숫자>0이면 안전 cap
PAGE_SIZE = 500
RESUME_CRAWL = True      # Drive checkpoint / efetch cache 재사용
REUSE_COMPLETED_STEPS = True  # Drive에 완성 산출물이 있으면 완료 단계는 건너뜀
RUN_DEEP_ORCID_FOLLOWUP = True
MAX_AUTHORS = 50         # 심층 ORCID 추적 대상 수 (전체 지도에는 제한 없음)
MAX_PER_AUTHOR = 150
TOPIC = ""               # 예: "stroke OR cerebrovascular"; 전체면 ""
TOPIC_THRESHOLD = 0.30
DRIVE_ROOT = Path("/content/drive/MyDrive/Paper_AI_Assistant/SKKU_PubMed")
NCBI_EMAIL = os.environ.get("NCBI_EMAIL") or input("NCBI email: ").strip()
NCBI_API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # 있으면 자동 사용

if not NCBI_EMAIL:
    raise ValueError("NCBI_EMAIL이 필요합니다.")

# ===== Google Drive 영구 저장 =====
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LINEAGE_DIR = DRIVE_ROOT / "skku_pubmed_lineage"
SEED_MAP_DIR = DRIVE_ROOT / "skku_pubmed_seed_map"
FOLLOWUP_DIR = DRIVE_ROOT / "skku_pubmed_followup"
for _p in [LINEAGE_DIR, SEED_MAP_DIR, FOLLOWUP_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

print(f"💾 Google Drive output root: {DRIVE_ROOT}")

# ===== GitHub 최신 코드 동기화 =====
REPO = Path("/content/Paper_AI_Assistant")
URL = "https://github.com/kimtk94/Paper_AI_Assistant.git"

if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "-q", URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", "origin/main"], check=True)

os.chdir(REPO)

# 이전 Colab runtime의 crawl checkpoint가 있으면 Drive로 1회 이관
legacy_lineage = REPO / "outputs/skku_pubmed_lineage"
for _name in ["crawl_checkpoint.json", "crawl_windows.json"]:
    _src = legacy_lineage / _name
    _dst = LINEAGE_DIR / _name
    if _src.exists() and not _dst.exists():
        shutil.copy2(_src, _dst)
        print(f"♻️ migrated legacy checkpoint: {_src} -> {_dst}")

# ===== 실행 전 문법 검사 =====
for script in ["src/skku_pubmed_lineage.py", "src/skku_pubmed_annotation.py", "src/skku_pubmed_seed_profiles.py", "src/skku_pubmed_author_followup.py", "src/skku_pubmed_researcher_network.py", "src/skku_pubmed_research_communities.py"]:
    subprocess.run([sys.executable, "-m", "py_compile", script], check=True)
print("✅ Python syntax check passed")

env = os.environ.copy()
env["NCBI_EMAIL"] = NCBI_EMAIL
if NCBI_API_KEY:
    env["NCBI_API_KEY"] = NCBI_API_KEY

def run_stream(cmd, label, env=None, tail_lines=80):
    """Run a long command while streaming logs; show the real failure tail."""
    print(f"▶ {label}")
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    tail = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        tail.append(line.rstrip())
        if len(tail) > tail_lines:
            tail.pop(0)
    rc = proc.wait()
    if rc != 0:
        diagnostic = "\n".join(tail[-tail_lines:])
        raise RuntimeError(
            f"{label} failed with exit code {rc}.\n"
            f"----- last {min(len(tail), tail_lines)} log lines -----\n"
            f"{diagnostic}"
        )
    return rc

def read_json_if_exists(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

# ===== STEP 1: SKKU PubMed graph =====
cmd1 = [
    sys.executable, "src/skku_pubmed_lineage.py",
    "--start-year", str(START_YEAR),
    "--end-year", str(END_YEAR),
    "--max-results", str(MAX_RESULTS),
    "--topic-threshold", str(TOPIC_THRESHOLD),
    "--skip-citations",
    "--output-dir", str(LINEAGE_DIR),
]
if FULL_CRAWL:
    cmd1 += ["--all-results", "--page-size", str(PAGE_SIZE)]
    if RESUME_CRAWL:
        cmd1 += ["--resume"]
if TOPIC.strip():
    cmd1 += ["--topic", TOPIC.strip()]

lineage_summary = read_json_if_exists(LINEAGE_DIR / "summary.json")
step1_complete = (
    REUSE_COMPLETED_STEPS
    and (LINEAGE_DIR / "papers.json").exists()
    and lineage_summary is not None
    and int(lineage_summary.get("retrieved_pmids", 0)) > 0
    and int(lineage_summary.get("verified_papers", 0)) > 0
    and (
        not FULL_CRAWL
        or int(lineage_summary.get("retrieved_pmids", 0))
           == int(lineage_summary.get("pubmed_total_hits", -1))
    )
    and f"{START_YEAR}:{END_YEAR}[dp]" in str(lineage_summary.get("query", ""))
    and (not TOPIC.strip() or TOPIC.strip() in str(lineage_summary.get("query", "")))
)

print("\n=== STEP 1/5: SKKU PubMed full crawl ===")
if step1_complete:
    print(
        "⏭️ STEP 1 reused from Drive: "
        f"{lineage_summary.get('verified_papers', 0):,} verified papers / "
        f"{lineage_summary.get('retrieved_pmids', 0):,} PMIDs"
    )
else:
    run_stream(cmd1, "STEP 1/5 SKKU PubMed full crawl", env=env)
    lineage_summary = read_json_if_exists(LINEAGE_DIR / "summary.json")

# ===== STEP 2: full seed corpus -> researcher profiles =====
print("\n=== STEP 2/5: Full-corpus researcher profiles ===")
seed_summary = read_json_if_exists(SEED_MAP_DIR / "seed_profile_summary.json")
expected_seed_papers = int((lineage_summary or {}).get("verified_papers", 0))
step2_complete = (
    REUSE_COMPLETED_STEPS
    and seed_summary is not None
    and (SEED_MAP_DIR / "researcher_summary.json").exists()
    and (SEED_MAP_DIR / "lineage_papers.json").exists()
    and int(seed_summary.get("seed_papers", -1)) == expected_seed_papers
)
if step2_complete:
    print(
        "⏭️ STEP 2 reused from Drive: "
        f"{seed_summary.get('researchers', 0):,} researchers"
    )
else:
    run_stream([
        sys.executable, "src/skku_pubmed_seed_profiles.py",
        "--papers-json", str(LINEAGE_DIR / "papers.json"),
        "--output-dir", str(SEED_MAP_DIR),
    ], "STEP 2/5 Full-corpus researcher profiles", env=env)
    seed_summary = read_json_if_exists(SEED_MAP_DIR / "seed_profile_summary.json")

# ===== STEP 3: scalable observed-edge researcher network =====
print("\n=== STEP 3/5: SKKU-wide researcher network ===")
network_summary = read_json_if_exists(SEED_MAP_DIR / "researcher_network_summary.json")
expected_researchers = int((seed_summary or {}).get("researchers", 0))
step3_complete = (
    REUSE_COMPLETED_STEPS
    and network_summary is not None
    and (SEED_MAP_DIR / "researcher_network_nodes.json").exists()
    and (SEED_MAP_DIR / "researcher_network_edges.json").exists()
    and int(network_summary.get("researchers", -1)) == expected_researchers
    and network_summary.get("thematic_enabled") is False
)
if step3_complete:
    print(
        "⏭️ STEP 3 reused from Drive: "
        f"{network_summary.get('researchers', 0):,} researchers / "
        f"{network_summary.get('network_edges', 0):,} edges"
    )
else:
    run_stream([
        sys.executable, "src/skku_pubmed_researcher_network.py",
        "--input-dir", str(SEED_MAP_DIR),
        "--skip-thematic",
        "--topic-threshold", str(TOPIC_THRESHOLD),
    ], "STEP 3/5 SKKU-wide researcher network", env=env)
    network_summary = read_json_if_exists(SEED_MAP_DIR / "researcher_network_summary.json")

# ===== STEP 4: research communities =====
print("\n=== STEP 4/5: SKKU-wide research communities ===")
community_summary = read_json_if_exists(SEED_MAP_DIR / "research_community_summary.json")
step4_complete = (
    REUSE_COMPLETED_STEPS
    and community_summary is not None
    and (SEED_MAP_DIR / "research_communities.json").exists()
    and (SEED_MAP_DIR / "researcher_community_membership.json").exists()
    and int(community_summary.get("researchers", -1))
        == int((network_summary or {}).get("researchers", 0))
)
if step4_complete:
    print(
        "⏭️ STEP 4 reused from Drive: "
        f"{community_summary.get('communities', 0):,} communities"
    )
else:
    run_stream([
        sys.executable, "src/skku_pubmed_research_communities.py",
        "--input-dir", str(SEED_MAP_DIR),
        "--min-edge-score", "0.65",
        "--strong-edge-score", "0.85",
    ], "STEP 4/5 SKKU-wide research communities", env=env)

# ===== STEP 5: optional ORCID deep follow-up =====
if RUN_DEEP_ORCID_FOLLOWUP:
    print("\n=== STEP 5/5: ORCID deep continuation ===")
    cmd2 = [
        sys.executable, "src/skku_pubmed_author_followup.py",
        "--seed-json", str(LINEAGE_DIR / "papers.json"),
        "--start-year", "2002",
        "--end-year", str(END_YEAR),
        "--max-authors", str(MAX_AUTHORS),
        "--max-per-author", str(MAX_PER_AUTHOR),
        "--with-citations",
        "--max-citation-checks", "300",
        "--output-dir", str(FOLLOWUP_DIR),
    ]
    run_stream(cmd2, "STEP 5/5 ORCID deep continuation", env=env)
else:
    print("\n=== STEP 5/5: ORCID deep continuation skipped ===")

# ===== 빈 CSV도 안전하게 읽기 =====
def safe_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

papers = safe_csv(str(LINEAGE_DIR / "papers.csv"))
edges = safe_csv(str(LINEAGE_DIR / "edges.csv"))
full_researchers = safe_csv(str(SEED_MAP_DIR / "researcher_summary.csv"))
researcher_network = safe_csv(str(SEED_MAP_DIR / "researcher_network_edges.csv"))
researcher_nodes = safe_csv(str(SEED_MAP_DIR / "researcher_network_nodes.csv"))
communities = safe_csv(str(SEED_MAP_DIR / "research_communities.csv"))
community_members = safe_csv(str(SEED_MAP_DIR / "researcher_community_membership.csv"))
community_edges = safe_csv(str(SEED_MAP_DIR / "community_edges.csv"))

registry = safe_csv(str(FOLLOWUP_DIR / "author_registry.csv"))
followups = safe_csv(str(FOLLOWUP_DIR / "lineage_papers.csv"))
continuation = safe_csv(str(FOLLOWUP_DIR / "continuation_edges.csv"))
lineage_links = safe_csv(str(FOLLOWUP_DIR / "lineage_edges.csv"))
annotations = safe_csv(str(FOLLOWUP_DIR / "paper_annotations.csv"))
researchers = safe_csv(str(FOLLOWUP_DIR / "researcher_summary.csv"))

print("\n================ RESULT ================")
print(f"SKKU verified papers : {len(papers):,}")
print(f"SKKU graph edges     : {len(edges):,}")
print(f"Full-map researchers : {len(full_researchers):,}")
print(f"Full-map links       : {len(researcher_network):,}")
print(f"Research communities : {len(communities):,}")
print(f"Deep tracked authors : {len(registry):,}")
print(f"Follow-up papers     : {len(followups):,}")
print(f"Continuation edges   : {len(continuation):,}")
print(f"Research lineage     : {len(lineage_links):,}")
print(f"Annotated papers     : {len(annotations):,}")
print(f"Researcher nodes     : {len(researcher_nodes):,}")
print(f"Deep trajectories    : {len(researchers):,}")

if not edges.empty and "relation" in edges:
    print("\n[SKKU edge types]")
    display(edges["relation"].value_counts().rename_axis("relation").reset_index(name="count"))

if not full_researchers.empty:
    print("\n[Full-corpus SKKU researchers]")
    cols = [c for c in [
        "name", "orcid", "confidence", "paper_count", "first_year", "last_year",
        "top_diseases", "top_methods", "top_data_types", "stage_path"
    ] if c in full_researchers.columns]
    display(full_researchers[cols].head(200))

if not registry.empty:
    print("\n[Deep ORCID tracked researchers]")
    cols = [c for c in ["name", "orcid", "confidence", "seed_pmids"] if c in registry.columns]
    display(registry[cols].head(30))

if not followups.empty:
    print("\n[Researcher publication continuation]")
    cols = [c for c in [
        "year", "tracked_authors", "title", "disease_terms", "methods",
        "data_types", "research_stage", "research_question",
        "journal", "skku_current", "pubmed_url"
    ] if c in followups.columns]
    display(followups[cols].sort_values(["tracked_authors", "year"], ascending=[True, True]).head(100))

if not annotations.empty:
    print("\n[Paper research profiles]")
    cols = [c for c in [
        "pmid", "disease_terms", "methods", "data_types",
        "research_stage", "research_question"
    ] if c in annotations.columns]
    display(annotations[cols].head(100))

if not researchers.empty:
    print("\n[Researcher trajectories]")
    cols = [c for c in [
        "name", "orcid", "paper_count", "first_year", "last_year",
        "top_diseases", "top_methods", "top_data_types",
        "stage_path", "strong_lineage_count", "trajectory_summary"
    ] if c in researchers.columns]
    display(researchers[cols].head(100))

if not lineage_links.empty:
    print("\n[Research lineage: direct citation > ORCID-only]")
    cols = [c for c in [
        "score", "relation", "source", "target", "tracked_authors",
        "source_stage", "target_stage", "progression", "evidence"
    ] if c in lineage_links.columns]
    display(lineage_links[cols].sort_values("score", ascending=False).head(100))

if not researcher_network.empty:
    print("\n[Researcher network: collaboration / citation / thematic overlap]")
    cols = [c for c in [
        "score", "relation", "source_name", "target_name", "shared_papers",
        "direct_citations", "topic_similarity", "shared_diseases",
        "shared_methods", "shared_data_types", "evidence"
    ] if c in researcher_network.columns]
    display(researcher_network[cols].sort_values("score", ascending=False).head(100))

if not communities.empty:
    print("\n[Research communities]")
    cols = [c for c in [
        "community_id", "label", "researcher_count", "hub_researcher",
        "total_papers", "first_year", "last_year", "top_diseases",
        "top_methods", "top_data_types", "stage_path",
        "strong_edge_count", "evidence_summary"
    ] if c in communities.columns]
    display(communities[cols].head(100))

if not community_members.empty:
    print("\n[Community membership / hub researchers]")
    cols = [c for c in [
        "community_id", "researcher_name", "orcid", "paper_count",
        "weighted_degree", "within_community_degree", "role"
    ] if c in community_members.columns]
    display(community_members[cols].head(200))

if not continuation.empty:
    print("\n[Raw continuation/citation edges]")
    display(continuation.head(100))

# ===== 결과 ZIP =====
zip_base = str(DRIVE_ROOT / "SKKU_PubMed_Lineage_results")
tmp_bundle = Path("/content/SKKU_PubMed_bundle")
if tmp_bundle.exists():
    shutil.rmtree(tmp_bundle)
tmp_bundle.mkdir(parents=True, exist_ok=True)

for src_dir in [LINEAGE_DIR, SEED_MAP_DIR, FOLLOWUP_DIR]:
    if src_dir.exists():
        shutil.copytree(
            src_dir,
            tmp_bundle / src_dir.name,
            dirs_exist_ok=True,
            ignore=shutil.ignore_patterns("efetch_cache"),
        )

shutil.make_archive(zip_base, "zip", root_dir=tmp_bundle)
print(f"\n📦 ZIP: {zip_base}.zip")
print(f"🌐 Interactive graph: {LINEAGE_DIR / 'lineage.html'}")
print(f"📄 Research chains: {FOLLOWUP_DIR / 'continuation.md'}")
print(f"🧑‍🔬 SKKU-wide researcher network: {SEED_MAP_DIR / 'researcher_network.html'}")
print(f"🧩 SKKU-wide research community map: {SEED_MAP_DIR / 'research_community_map.html'}")
print(f"🧭 Deep ORCID trajectory HTML: {FOLLOWUP_DIR / 'research_trajectory.html'}")
print(f"💾 Persistent efetch cache: {LINEAGE_DIR / 'efetch_cache'}")

# Colab에서 interactive HTML 바로 표시
html_path = LINEAGE_DIR / "lineage.html"
if html_path.exists():
    display(HTML("<b>완료:</b> 왼쪽 Files에서 <code>lineage.html</code> 또는 결과 ZIP을 열면 됩니다."))
